In [ ]:
import requests
import pandas as pd
from datetime import datetime

print("="*70)
print("FETCHING REAL MALICIOUS URLS FROM SECURITY FEEDS")
print("="*70)

all_malicious = []


# 1. OPENPHISH - Live Phishing Feed (Most Reliable)

print("\n[1/3] OpenPhish (live phishing)...")
try:
    r = requests.get("https://openphish.com/feed.txt", timeout=15)
    if r.status_code == 200:
        urls = [u.strip() for u in r.text.strip().split('\n') if u.strip()]
        all_malicious.extend([{'url': u, 'source': 'openphish', 'type': 'phishing'} for u in urls[:50]])
        print(f"  ✓ Got {len(urls[:50])} phishing URLs")
        print(f"  Sample: {urls[0] if urls else 'None'}")
    else:
        print(f"  ✗ HTTP {r.status_code}")
except Exception as e:
    print(f"  ✗ Error: {e}")

# 2. URLHAUS - Recent Malware (Last 48 hours)

print("\n[2/3] URLhaus (recent malware)...")
try:
    r = requests.get("https://urlhaus-api.abuse.ch/v1/urls/recent/", 
                     timeout=15,
                     headers={"User-Agent": "URLClassifier/1.0"})
    if r.status_code == 200:
        data = r.json()
        urls = [item['url'] for item in data.get('urls', [])[:50] if item.get('url')]
        all_malicious.extend([{'url': u, 'source': 'urlhaus', 'type': 'malware'} for u in urls])
        print(f"  ✓ Got {len(urls)} malware URLs")
        if urls:
            print(f"  Sample: {urls[0]}")
    else:
        print(f"  ✗ HTTP {r.status_code}")
except Exception as e:
    print(f"  ✗ Error: {e}")


# 3. PHISHTANK - Verified Phishing (if available)

print("\n[3/3] PhishTank (verified phishing)...")
try:
    r = requests.get("http://data.phishtank.com/data/online-valid.csv", 
                     timeout=30)
    if r.status_code == 200:
        # Save and parse CSV
        with open('phishtank_temp.csv', 'wb') as f:
            f.write(r.content)
        
        df = pd.read_csv('phishtank_temp.csv')
        if 'url' in df.columns:
            urls = df['url'].head(50).tolist()
            all_malicious.extend([{'url': u, 'source': 'phishtank', 'type': 'phishing'} for u in urls])
            print(f"  ✓ Got {len(urls)} phishing URLs")
        
        import os
        os.remove('phishtank_temp.csv')
    else:
        print(f"  ✗ HTTP {r.status_code}")
except Exception as e:
    print(f"  ✗ Error: {e}")


# RESULTS

print("\n" + "="*70)
print("SUMMARY")
print("="*70)

if not all_malicious:
    print("❌ No malicious URLs collected!")
    print("Try running again - feeds may be temporarily unavailable")
else:
    df_malicious = pd.DataFrame(all_malicious)
    print(f"\n✓ Collected {len(df_malicious)} real malicious URLs")
    print("\nBreakdown:")
    print(df_malicious.groupby(['source', 'type']).size())
    
    # Save
    df_malicious.to_csv('real_malicious_urls.csv', index=False)
    print(f"\n✓ Saved to real_malicious_urls.csv")
    
    # Show samples
    print("\n" + "="*70)
    print("SAMPLE MALICIOUS URLS (first 10)")
    print("="*70)
    for i, row in df_malicious.head(10).iterrows():
        print(f"[{row['type']:<10}] {row['url'][:70]}")



FETCHING REAL MALICIOUS URLS FROM SECURITY FEEDS

[1/3] OpenPhish (live phishing)...
  ✓ Got 50 phishing URLs
  Sample: https://cn.plhhovyj.com/home/register

[2/3] URLhaus (recent malware)...
  ✗ HTTP 401

[3/3] PhishTank (verified phishing)...
  ✓ Got 50 phishing URLs

SUMMARY

✓ Collected 100 real malicious URLs

Breakdown:
source     type    
openphish  phishing    50
phishtank  phishing    50
dtype: int64

✓ Saved to real_malicious_urls.csv

SAMPLE MALICIOUS URLS (first 10)
[phishing  ] https://cn.plhhovyj.com/home/register
[phishing  ] http://jixunjidian_com.51tiantian.com/
[phishing  ] http://security-base-pro.daftpage.com/
[phishing  ] http://roblox.com.ml/communities/5549878422/BrokenWand
[phishing  ] http://b45030.com/poker/125
[phishing  ] http://b45030.com/fish/29
[phishing  ] http://b45030.com/fish
[phishing  ] https://mycssch.tithelysetup.com/mycss
[phishing  ] http://shiva-priyanshu.github.io/netflixclone.github.io
[phishing  ] http://www.insttargram.blogspot.com/

READY